In [1]:
print("Hello")

Hello


In [2]:
import requests
import smtplib
from email.mime.text import MIMEText
from requests.auth import HTTPBasicAuth

# Azure DevOps Settings
ORG = "1Wan"
PROJECT = "SWAN"
PAT = "YOUR_PAT_TOKEN"

# Email Settings
SMTP_SERVER = "smtp.office365.com"
SMTP_PORT = 587

SENDER = "your_email@microsoft.com"
PASSWORD = "your_password_or_app_password"

RECIPIENT = "your_email@microsoft.com"

URL = (
    f"https://dev.azure.com/{ORG}/{PROJECT}"
    "/_apis/build/builds"
    "?statusFilter=inProgress"
    "&api-version=7.1-preview.7"
)

def get_running_pipelines():
    response = requests.get(
        URL,
        auth=HTTPBasicAuth("", PAT)
    )

    response.raise_for_status()

    data = response.json()

    return data.get("count", 0)

def send_email():
    subject = "SWAN Alert - No Pipelines Running"

    body = """
No Azure DevOps pipelines are currently running.

Project: SWAN
Organization: 1Wan

Please verify whether jobs should be running.
"""

    msg = MIMEText(body)

    msg["Subject"] = subject
    msg["From"] = SENDER
    msg["To"] = RECIPIENT

    with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
        server.starttls()
        server.login(SENDER, PASSWORD)
        server.send_message(msg)

def main():
    running = get_running_pipelines()

    print(f"Running pipelines: {running}")

    if running == 0:
        send_email()
        print("Email sent")
    else:
        print("Pipelines active")

if __name__ == "__main__":
    main()

C:\Users\v-yaalam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


JSONDecodeError: Expecting value: line 3 column 1 (char 4)

In [ ]:
import os

STATE_FILE = "alert_sent.txt"

if running == 0:
    if not os.path.exists(STATE_FILE):
        send_email()
        open(STATE_FILE, "w").close()
else:
    if os.path.exists(STATE_FILE):
        os.remove(STATE_FILE)

Yes. Do not scrape the Azure DevOps webpage directly. The better approach is to use the Azure DevOps REST API and check for pipelines whose status is inProgress. This is more reliable and won't break if the UI changes.

High-Level Design

Every 30 minutes:

Query Azure DevOps Builds API.
Check if any builds have status inProgress.
If 0 pipelines running, send email notification.
Optionally avoid sending duplicate emails repeatedly.
Step 1: Create a PAT

Azure DevOps → User Settings → Personal Access Tokens

Grant:

Build (Read)
Project and Team (Read)

Save the PAT.

Step 2: Python Script
import requests
import smtplib
from email.mime.text import MIMEText
from requests.auth import HTTPBasicAuth

# Azure DevOps Settings
ORG = "1Wan"
PROJECT = "SWAN"
PAT = "YOUR_PAT_TOKEN"

# Email Settings
SMTP_SERVER = "smtp.office365.com"
SMTP_PORT = 587

SENDER = "your_email@microsoft.com"
PASSWORD = "your_password_or_app_password"

RECIPIENT = "your_email@microsoft.com"

URL = (
    f"https://dev.azure.com/{ORG}/{PROJECT}"
    "/_apis/build/builds"
    "?statusFilter=inProgress"
    "&api-version=7.1-preview.7"
)

def get_running_pipelines():
    response = requests.get(
        URL,
        auth=HTTPBasicAuth("", PAT)
    )

    response.raise_for_status()

    data = response.json()

    return data.get("count", 0)

def send_email():
    subject = "SWAN Alert - No Pipelines Running"

    body = """
No Azure DevOps pipelines are currently running.

Project: SWAN
Organization: 1Wan

Please verify whether jobs should be running.
"""

    msg = MIMEText(body)

    msg["Subject"] = subject
    msg["From"] = SENDER
    msg["To"] = RECIPIENT

    with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
        server.starttls()
        server.login(SENDER, PASSWORD)
        server.send_message(msg)

def main():
    running = get_running_pipelines()

    print(f"Running pipelines: {running}")

    if running == 0:
        send_email()
        print("Email sent")
    else:
        print("Pipelines active")

if __name__ == "__main__":
    main()

Better Version (Prevent Spam)

If there are no pipelines for several hours, the above script will send an email every 30 minutes.

Create a small state file:

import os

STATE_FILE = "alert_sent.txt"

if running == 0:
    if not os.path.exists(STATE_FILE):
        send_email()
        open(STATE_FILE, "w").close()
else:
    if os.path.exists(STATE_FILE):
        os.remove(STATE_FILE)


This sends only one alert until pipelines start running again.

Schedule Every 30 Minutes
Linux
crontab -e


Add:

*/30 * * * * /usr/bin/python3 /home/user/check_pipelines.py

Windows Task Scheduler

Create Task

Trigger:

Repeat every: 30 minutes
Indefinitely


Action:

python.exe


Arguments:

C:\Scripts\check_pipelines.py

Even Better: Teams Notification

Since you're working in a Microsoft environment, sending a Teams message is often more useful than email.

Using a Teams Incoming Webhook:

import requests

webhook_url = "YOUR_TEAMS_WEBHOOK"

payload = {
    "text": "🚨 SWAN Alert: No Azure DevOps pipelines currently running."
}

requests.post(webhook_url, json=payload)

Enterprise-Grade Option (Recommended)

Since you're already using Azure DevOps, the cleanest solution is:

Azure Function or Automation Account
Run every 30 minutes
Query ADO Builds API
Send:
Outlook email
Teams notification
Adaptive Card

Benefits:

No laptop needs to stay on
Runs 24x7
Centralized monitoring
Easy auditing

For your SWAN project specifically, I'd recommend checking all builds currently in progress via:

GET https://dev.azure.com/1Wan/SWAN/_apis/build/builds?statusFilter=inProgress&api-version=7.1


and alert when the returned count is 0. This is the most reliable approach.